# 14 · Urban Analysis

Comprehensive urban mapping: buildings, growth, roads, infrastructure.

## Contents
1. Building footprint extraction
2. Urban growth monitoring
3. Road network extraction
4. Infrastructure monitoring
5. Population density estimation

## 1 · Multi-City Building Extraction

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

CITIES = {
    "Chicago": (-87.7, 41.8, -87.5, 41.9),
    "London":  (-0.2, 51.4, 0.0, 51.6),
    "Tokyo":   (139.6, 35.6, 139.8, 35.8),
}

print("Running building footprint extraction for 3 cities...")
results = {}
for city, bbox in CITIES.items():
    result = client.pipeline("building_footprints", bbox=bbox,
                              output_dir=f"./results/urban/{city.lower()}/",
                              date="2024-06")
    status = "[OK]" if result.success else "[FAIL]"
    print(f"  {status} {city}: {result.stats if result.success else result.error}")
    results[city] = result

## 2 · Urban Growth Analysis

In [ ]:
result = client.pipeline(
    "urban_growth",
    bbox=(-87.7, 41.8, -87.5, 41.9),
    output_dir="./results/urban/chicago_growth/",
    date_before="2000-01",
    date_after="2024-01",
)
print(f"Urban growth (2000->2024): {'[OK]' if result.success else '[FAIL]'}")
if result.success:
    print(f"  Output: {result.output_path}")

## 3 · Road Network Extraction

In [ ]:
result = client.pipeline(
    "road_extraction",
    bbox=(-87.65, 41.82, -87.62, 41.85),
    output_dir="./results/urban/roads/",
    date="2024-06",
)
print(f"Road extraction: {'[OK]' if result.success else '[FAIL]'}")

## 4 · Urban Density Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate building density analysis for a city grid
np.random.seed(42)
grid_size = 20
density_map = np.zeros((grid_size, grid_size))

# Create realistic urban density gradient (center = high, suburbs = low)
for i in range(grid_size):
    for j in range(grid_size):
        dist = np.sqrt((i - grid_size//2)**2 + (j - grid_size//2)**2)
        base = np.exp(-dist/6) * 80 + 5
        density_map[i, j] = base + np.random.normal(0, 3)

density_map = np.clip(density_map, 0, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Density heatmap
im = axes[0].imshow(density_map, cmap="hot_r", vmin=0, vmax=100)
plt.colorbar(im, ax=axes[0], label="Building density (%)")
axes[0].set_title("Building Density Map (500m grid)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("West -> East"); axes[0].set_ylabel("South -> North")

# Radial profile
center = grid_size // 2
distances = [np.sqrt((i-center)**2 + (j-center)**2) * 0.5
             for i in range(grid_size) for j in range(grid_size)]
densities = density_map.flatten()
axes[1].scatter(distances, densities, alpha=0.4, s=15, color="steelblue")

# Trend line
from numpy.polynomial import polynomial as P
coefs = P.polyfit(distances, densities, 3)
x_line = np.linspace(0, max(distances), 100)
y_line = P.polyval(x_line, coefs)
axes[1].plot(x_line, y_line, "r-", linewidth=2.5, label="Trend")
axes[1].set_xlabel("Distance from City Centre (km)", fontsize=12)
axes[1].set_ylabel("Building Coverage (%)", fontsize=12)
axes[1].set_title("Urban Density Gradient", fontsize=13, fontweight="bold")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./results/urban/density_analysis.png", dpi=150)
plt.show()
print(f"Peak density: {density_map.max():.0f}% (city centre)")
print(f"Edge density: {density_map[0, 0]:.0f}% (suburban)")

## 5 · EarthNets Urban Datasets

In [ ]:
from pygeovision.datasets.registry import dataset_registry
from pygeovision.datasets.benchmark import BenchmarkBuilder

# Top urban segmentation datasets
urban_ds = dataset_registry.filter(domain="urban", task="segmentation")
print(f"Urban segmentation datasets ({len(urban_ds)}):")
dataset_registry.print_table(urban_ds[:10])

# EarthNets recommended benchmark
builder = BenchmarkBuilder()
rec = builder.recommended_for_paper("segmentation")
print(f"\nPaper benchmark recommendation (segmentation):")
print(f"  Primary metric: {rec['primary_metric']}")
print(f"  Split: {rec['split']}")
for d in rec["datasets"][:3]:
    print(f"  - {d['name']} ({d['year']}, {d['resolution_m']}m, {d['n_samples']:,} samples)")